# 📊 Dissertation Data Collection Pipeline
## *From Tweets to Trades: Analysing Sentiment Signal and Information Network for Crypto Market Prediction*

---

This notebook collects **all data** required for the dissertation in one place:

| # | Data Stream | Source | Credentials Needed? |
|---|-----------|--------|--------------------|
| 1 | Market prices, volume, market cap | CoinGecko API | No |
| 2 | OHLCV candlestick data | Binance API | No |
| 3 | Reddit posts & comments | Arctic Shift + PRAW | PRAW optional |
| 4 | Twitter/X tweets | Published datasets (Kaggle) | Kaggle optional |

**Assets under study:**
- Bitcoin (BTC) — Established
- Ethereum (ETH) — Established
- Dogecoin (DOGE) — Meme coin
- Shiba Inu (SHIB) — Meme coin
- Tether (USDT) — Stablecoin control

**Time frame:** 1 January 2020 – 31 December 2024

**Instructions:** Run each section in order. At the end, download the collected data as a zip file.

---

## 0. Setup & Dependencies

In [ ]:
# Install any missing packages.
!pip install -q requests pandas numpy praw

import requests
import pandas as pd
import numpy as np
import time
import os
import json
import re
import glob
from datetime import datetime, timezone, timedelta
from google.colab import files

print(f"pandas {pd.__version__}, numpy {np.__version__}")
print("All dependencies loaded successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 4.8 MB/s eta 0:00:00
pandas 2.2.2, numpy 2.0.2
All dependencies loaded successfully.


In [ ]:
# Create the project folder structure.
DIRS = [
    "data/raw/market",
    "data/raw/sentiment/reddit",
    "data/raw/sentiment/twitter/published",
    "data/raw/sentiment/news",
    "data/processed",
    "data/merged",
    "docs"
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)
    print(f"  Created: {d}/")

print("\nFolder structure ready.")

  Created: data/raw/market/
  Created: data/raw/sentiment/reddit/
  Created: data/raw/sentiment/twitter/published/
  Created: data/raw/sentiment/news/
  Created: data/processed/
  Created: data/merged/
  Created: docs/

Folder structure ready.


In [ ]:
# =============================================================
# SHARED CONFIGURATION
# =============================================================

# Time frame for the study.
START_DATE = datetime(2020, 1, 1, tzinfo=timezone.utc)
END_DATE   = datetime(2024, 12, 31, 23, 59, 59, tzinfo=timezone.utc)

START_TS = int(START_DATE.timestamp())
END_TS   = int(END_DATE.timestamp())

# Asset definitions.
ASSETS = {
    "bitcoin":   {"coingecko_id": "bitcoin",    "symbol": "BTC",  "category": "established",         "binance": "BTCUSDT"},
    "ethereum":  {"coingecko_id": "ethereum",   "symbol": "ETH",  "category": "established",         "binance": "ETHUSDT"},
    "dogecoin":  {"coingecko_id": "dogecoin",   "symbol": "DOGE", "category": "meme_coin",           "binance": "DOGEUSDT"},
    "shiba_inu": {"coingecko_id": "shiba-inu",  "symbol": "SHIB", "category": "meme_coin",           "binance": "SHIBUSDT"},
    "tether":    {"coingecko_id": "tether",     "symbol": "USDT", "category": "stablecoin_control",  "binance": None},
}

# Keywords for detecting asset mentions in social media text.
ASSET_KEYWORDS = {
    "bitcoin":   ["bitcoin", "btc", "$btc", "#bitcoin"],
    "ethereum":  ["ethereum", "ether", "eth", "$eth", "#ethereum"],
    "dogecoin":  ["dogecoin", "doge", "$doge", "#dogecoin"],
    "shiba_inu": ["shiba", "shib", "$shib", "#shibarmy", "shiba inu", "shibainu"],
    "tether":    ["tether", "usdt", "$usdt"],
}

def detect_asset_mentions(text):
    """Detect which of the 5 study assets are mentioned in a text."""
    if not text or not isinstance(text, str):
        return []
    text_lower = text.lower()
    mentioned = []
    for asset, keywords in ASSET_KEYWORDS.items():
        for kw in keywords:
            if kw.startswith(("$", "#")):
                if kw in text_lower:
                    mentioned.append(asset); break
            elif len(kw) <= 4:
                if re.search(r'\b' + re.escape(kw) + r'\b', text_lower):
                    mentioned.append(asset); break
            else:
                if kw in text_lower:
                    mentioned.append(asset); break
    return mentioned

def clean_text(text):
    """Basic text cleaning for social media content."""
    if not text or not isinstance(text, str):
        return ""
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\[deleted\]|\[removed\]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Configuration loaded.")
print(f"Study period: {START_DATE.date()} to {END_DATE.date()}")
print(f"Assets: {', '.join(f'{v["symbol"]} ({k})' for k, v in ASSETS.items())}")

Configuration loaded.
Study period: 2020-01-01 to 2024-12-31
Assets: BTC (bitcoin), ETH (ethereum), DOGE (dogecoin), SHIB (shiba_inu), USDT (tether)


## 📈 Market Data: CryptoCompare (Daily OHLCV, Price, Volume)

This section fetches daily Open-High-Low-Close-Volume (OHLCV) market data from the **CryptoCompare** API (https://min-api.cryptocompare.com). CryptoCompare was selected as the primary market data source because it provides full unrestricted historical data going back to each coin's launch date, requires no API key for basic daily endpoints, and has no regional access restrictions.

**Data collected per asset per day:**
- Open, High, Low, Close prices (USD)
- Trading volume (in both USD and base asset terms)
- Derived features: daily returns, log returns, 7-day and 30-day rolling volatility, intraday range

**No API key or credentials required.**

In [ ]:
# =============================================================
# MARKET DATA: CryptoCompare (no regional blocks, no key needed
# for daily historical data, full history back to coin launch)
# =============================================================

def fetch_cryptocompare_daily(fsym, tsym="USD", limit=2000, toTs=None):
    """
    Fetch daily OHLCV data from CryptoCompare's free API.
    Returns up to 2000 days per request. No API key needed.
    """
    url = "https://min-api.cryptocompare.com/data/v2/histoday"
    params = {"fsym": fsym, "tsym": tsym, "limit": limit}
    if toTs:
        params["toTs"] = toTs
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    if data.get("Response") == "Error":
        raise Exception(data.get("Message", "Unknown CryptoCompare error"))
    return data["Data"]["Data"]

def collect_full_history(fsym, start_date, end_date, tsym="USD"):
    """
    Collect full daily history by paginating backwards from end_date.
    CryptoCompare returns up to 2000 days per call, so for 5 years
    we may need multiple calls.
    """
    all_records = []
    current_end_ts = int(end_date.timestamp())
    start_ts = int(start_date.timestamp())

    while True:
        records = fetch_cryptocompare_daily(fsym, tsym, limit=2000, toTs=current_end_ts)
        if not records:
            break

        # Filter to only include records within our date range.
        filtered = [r for r in records if r["time"] >= start_ts and r["time"] <= int(end_date.timestamp())]
        all_records.extend(filtered)

        # Find the earliest timestamp in this batch.
        earliest = min(r["time"] for r in records)
        if earliest <= start_ts:
            break  # We have reached or passed the start date.

        # Move the window back.
        current_end_ts = earliest - 1
        time.sleep(1)

    # Remove duplicates and sort.
    seen = set()
    unique = []
    for r in all_records:
        if r["time"] not in seen:
            seen.add(r["time"])
            unique.append(r)
    unique.sort(key=lambda x: x["time"])
    return unique

def parse_cryptocompare(records, asset_name, symbol, category):
    """Parse CryptoCompare daily records into a clean DataFrame."""
    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["time"], unit="s")
    df["asset"]    = asset_name
    df["symbol"]   = symbol
    df["category"] = category
    df["price_usd"]       = df["close"]
    df["volume_usd"]      = df["volumeto"]    # Volume in USD terms
    df["daily_range_pct"] = ((df["high"] - df["low"]) / df["open"].replace(0, np.nan)) * 100
    df["daily_return_pct"] = df["close"].pct_change() * 100
    df["log_return"]       = np.log(df["close"] / df["close"].shift(1))
    df["volatility_7d"]    = df["log_return"].rolling(7).std()
    df["volatility_30d"]   = df["log_return"].rolling(30).std()
    df["volume_change_pct"] = df["volumeto"].pct_change() * 100

    # Filter out rows with zero volume (coin didn't exist yet).
    df = df[df["volumeto"] > 0]

    out_cols = ["date","asset","symbol","category","open","high","low","close",
                "price_usd","volume_usd","volumefrom","daily_range_pct",
                "daily_return_pct","log_return","volatility_7d",
                "volatility_30d","volume_change_pct"]
    return df[out_cols].sort_values("date").reset_index(drop=True)


# ---- Run collection ----
print("=" * 60)
print("MARKET DATA COLLECTION (CryptoCompare - no key needed)")
print("=" * 60)

CC_ASSETS = {
    "bitcoin":   {"fsym": "BTC",  "category": "established"},
    "ethereum":  {"fsym": "ETH",  "category": "established"},
    "dogecoin":  {"fsym": "DOGE", "category": "meme_coin"},
    "shiba_inu": {"fsym": "SHIB", "category": "meme_coin"},
    "tether":    {"fsym": "USDT", "category": "stablecoin_control"},
}

all_frames = []

for name, info in CC_ASSETS.items():
    print(f"\n>>> {name} ({info['fsym']})...")
    try:
        records = collect_full_history(info["fsym"], START_DATE, END_DATE)
        df = parse_cryptocompare(records, name, info["fsym"], info["category"])
        path = f"data/raw/market/{name}_daily.csv"
        df.to_csv(path, index=False)
        all_frames.append(df)
        print(f"    ✓ {len(df)} days  |  {df['date'].min().date()} to {df['date'].max().date()}")
        print(f"    Saved: {path}")
    except Exception as e:
        print(f"    ✗ ERROR: {e}")
    time.sleep(2)

# Master file.
if all_frames:
    master = pd.concat(all_frames, ignore_index=True)
    master.to_csv("data/raw/market/all_assets_daily.csv", index=False)
    print(f"\n{'=' * 60}")
    print(f"✓ Master file: data/raw/market/all_assets_daily.csv ({len(master)} rows)")
    print(f"{'=' * 60}")

    summary = master.groupby("asset").agg(
        days=("date", "count"),
        start=("date", "min"),
        end=("date", "max"),
        mean_price=("price_usd", "mean"),
        mean_volume=("volume_usd", "mean"),
    ).round(2)
    display(summary)
    print("\nMarket data collection COMPLETE.")

MARKET DATA COLLECTION (CryptoCompare - no key needed)

>>> bitcoin (BTC)...
    ✓ 1827 days  |  2020-01-01 to 2024-12-31
    Saved: data/raw/market/bitcoin_daily.csv

>>> ethereum (ETH)...
    ✓ 1827 days  |  2020-01-01 to 2024-12-31
    Saved: data/raw/market/ethereum_daily.csv

>>> dogecoin (DOGE)...
    ✓ 1827 days  |  2020-01-01 to 2024-12-31
    Saved: data/raw/market/dogecoin_daily.csv

>>> shiba_inu (SHIB)...
    ✓ 1289 days  |  2021-06-22 to 2024-12-31
    Saved: data/raw/market/shiba_inu_daily.csv

>>> tether (USDT)...
    ✓ 1827 days  |  2020-01-01 to 2024-12-31
    Saved: data/raw/market/tether_daily.csv

✓ Master file: data/raw/market/all_assets_daily.csv (8597 rows)


,days,start,end,mean_price,mean_volume
asset,,,,,
bitcoin,1827,2020-01-01,2024-12-31,36314.44,1.422144e+09
dogecoin,1827,2020-01-01,2024-12-31,0.11,7.740770e+07
ethereum,1827,2020-01-01,2024-12-31,1982.30,9.163719e+08
shiba_inu,1289,2021-06-22,2024-12-31,0.00,5.211047e+07
tether,1827,2020-01-01,2024-12-31,1.00,2.126922e+07



Market data collection COMPLETE.


In [ ]:
def fetch_binance_klines(symbol, interval="1d", start_ms=None, end_ms=None):
    """Fetch all daily klines from Binance, paginating through 1000-record chunks."""
    url = "https://api.binance.com/api/v3/klines"
    all_klines = []
    current = start_ms
    while current < end_ms:
        params = {"symbol": symbol, "interval": interval,
                  "startTime": current, "endTime": end_ms, "limit": 1000}
        resp = requests.get(url, params=params, timeout=30)
        if resp.status_code == 429:
            print("    Rate limited. Waiting 30s...")
            time.sleep(30); continue
        resp.raise_for_status()
        klines = resp.json()
        if not klines: break
        all_klines.extend(klines)
        current = klines[-1][6] + 1  # Close time of last candle + 1ms.
        if len(klines) < 1000: break
        time.sleep(0.5)
    return all_klines

def parse_binance(klines, asset_name):
    """Parse Binance kline data into a clean OHLCV DataFrame."""
    cols = ["open_time_ms","open","high","low","close","vol_base",
            "close_time_ms","vol_quote","num_trades","taker_buy_base",
            "taker_buy_quote","_"]
    df = pd.DataFrame(klines, columns=cols)
    for c in ["open","high","low","close","vol_base","vol_quote","num_trades"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["date"] = pd.to_datetime(df["open_time_ms"], unit="ms").dt.date
    df["date"] = pd.to_datetime(df["date"])
    df["daily_range_pct"] = ((df["high"] - df["low"]) / df["open"]) * 100
    df["log_return"]      = np.log(df["close"] / df["close"].shift(1))
    df["volatility_7d"]   = df["log_return"].rolling(7).std()
    df["volatility_30d"]  = df["log_return"].rolling(30).std()
    df["asset"] = asset_name
    out_cols = ["date","asset","open","high","low","close","vol_base",
               "vol_quote","num_trades","daily_range_pct","log_return",
               "volatility_7d","volatility_30d"]
    return df[out_cols].sort_values("date").reset_index(drop=True)

In [ ]:
# ---- Run Binance collection ----
print("=" * 60)
print("BINANCE OHLCV DATA COLLECTION")
print("=" * 60)

start_ms = START_TS * 1000
end_ms   = END_TS * 1000
bn_frames = []

for name, info in ASSETS.items():
    pair = info.get("binance")
    if not pair:
        print(f"\n>>> {name}: No Binance pair (collected via CoinGecko only).")
        continue
    print(f"\n>>> {name} ({pair})...")
    try:
        klines = fetch_binance_klines(pair, "1d", start_ms, end_ms)
        df = parse_binance(klines, name)
        path = f"data/raw/market/{name}_binance_ohlcv.csv"
        df.to_csv(path, index=False)
        bn_frames.append(df)
        print(f"    {len(df)} days  |  {df['date'].min().date()} to {df['date'].max().date()}")
        print(f"    Saved: {path}")
    except Exception as e:
        print(f"    ERROR: {e}")
    time.sleep(1)

if bn_frames:
    master_bn = pd.concat(bn_frames, ignore_index=True)
    master_bn.to_csv("data/raw/market/all_assets_binance_ohlcv.csv", index=False)
    print(f"\nMaster file: data/raw/market/all_assets_binance_ohlcv.csv ({len(master_bn)} rows)")
    print("\nBinance collection COMPLETE.")

---
## 3. 🗣️ Reddit Sentiment Data (Arctic Shift API)

Collects historical Reddit posts from crypto subreddits using the **Arctic Shift** API (the successor to Pushshift). No credentials are required.

**Target subreddits:**
- Asset-specific: `r/Bitcoin`, `r/ethereum`, `r/dogecoin`, `r/SHIBArmy`
- General: `r/CryptoCurrency`, `r/CryptoMarkets`, `r/SatoshiStreetBets`

> **Note:** Arctic Shift may take some time for large subreddits. The `max_per_sub` parameter controls how many posts to collect per subreddit. Adjust as needed.

In [ ]:
def collect_arctic_shift(subreddit, start_epoch, end_epoch, max_records=50000):
    """
    Collect historical Reddit submissions from Arctic Shift API.
    Paginates through results using created_utc sorting.
    """
    base_url = "https://arctic-shift.photon-reddit.com/api/posts/search"

    records = []
    current_after = start_epoch
    req_count = 0

    while current_after < end_epoch and len(records) < max_records:
        params = {
            "subreddit": subreddit,
            "after": current_after,
            "before": end_epoch,
            "limit": 100,
            "sort": "asc",
            "sort_type": "created_utc"
        }
        try:
            resp = requests.get(base_url, params=params, timeout=30)
            if resp.status_code == 429:
                print("    Rate limited. Waiting 30s...")
                time.sleep(30); continue
            resp.raise_for_status()
            data = resp.json()
            results = data.get("data", [])
            if not results: break

            for item in results:
                created = item.get("created_utc", 0)
                full_text = f"{item.get('title', '')} {item.get('selftext', '')}"
                records.append({
                    "id": item.get("id", ""),
                    "created_utc": datetime.fromtimestamp(created, tz=timezone.utc).isoformat(),
                    "date": datetime.fromtimestamp(created, tz=timezone.utc).strftime("%Y-%m-%d"),
                    "subreddit": subreddit,
                    "title": item.get("title", ""),
                    "text": clean_text(item.get("selftext", "")),
                    "score": item.get("score", 0),
                    "num_comments": item.get("num_comments", 0),
                    "asset_mentions": json.dumps(detect_asset_mentions(full_text)),
                })

            current_after = results[-1].get("created_utc", current_after) + 1
            req_count += 1
            if req_count % 20 == 0:
                print(f"    r/{subreddit}: {len(records)} posts so far...")
            if len(results) < 100: break
            time.sleep(2)  # Rate limit.

        except requests.exceptions.RequestException as e:
            print(f"    Error: {e}")
            break

    return pd.DataFrame(records)

In [ ]:
# ---- Configure collection targets ----
# Adjust max_per_sub to control collection size.
# Larger values = more data but longer runtime.

REDDIT_TARGETS = {
    # Asset-specific subreddits.
    "bitcoin":   ["Bitcoin"],
    "ethereum":  ["ethereum"],
    "dogecoin":  ["dogecoin"],
    "shiba_inu": ["SHIBArmy"],
    # General crypto subreddits.
    "general":   ["CryptoCurrency", "CryptoMarkets", "SatoshiStreetBets"],
}

MAX_PER_SUB = 50000   # Max posts per subreddit. Reduce if short on time.

print(f"Will collect up to {MAX_PER_SUB} posts per subreddit.")
print(f"Target subreddits: {sum(len(v) for v in REDDIT_TARGETS.values())}")

Will collect up to 50000 posts per subreddit.
Target subreddits: 7


In [ ]:
# ---- Run Reddit collection ----
print("=" * 60)
print("REDDIT HISTORICAL DATA COLLECTION (Arctic Shift)")
print("=" * 60)

reddit_frames = []

for group, subs in REDDIT_TARGETS.items():
    for sub in subs:
        print(f"\n>>> r/{sub} (group: {group})...")
        df = collect_arctic_shift(sub, START_TS, END_TS, max_records=MAX_PER_SUB)
        if not df.empty:
            path = f"data/raw/sentiment/reddit/{group}_r{sub}_posts.csv"
            df.to_csv(path, index=False)
            reddit_frames.append(df)
            print(f"    {len(df)} posts  |  {df['date'].min()} to {df['date'].max()}")
            print(f"    Saved: {path}")
        else:
            print(f"    No data returned.")

if reddit_frames:
    master_reddit = pd.concat(reddit_frames, ignore_index=True)
    master_reddit = master_reddit.drop_duplicates(subset=["id"], keep="first")
    master_reddit.to_csv("data/raw/sentiment/reddit/all_reddit_posts.csv", index=False)
    print(f"\nMaster file: data/raw/sentiment/reddit/all_reddit_posts.csv")
    print(f"Total unique posts: {len(master_reddit)}")
    print(f"\nPer-subreddit breakdown:")
    print(master_reddit["subreddit"].value_counts().to_string())
    print("\nReddit collection COMPLETE.")

REDDIT HISTORICAL DATA COLLECTION (Arctic Shift)

>>> r/Bitcoin (group: bitcoin)...
    r/Bitcoin: 2000 posts so far...
    r/Bitcoin: 4000 posts so far...
    r/Bitcoin: 6000 posts so far...
    r/Bitcoin: 8000 posts so far...
    r/Bitcoin: 10000 posts so far...
    r/Bitcoin: 12000 posts so far...
    r/Bitcoin: 14000 posts so far...
    r/Bitcoin: 16000 posts so far...
    r/Bitcoin: 18000 posts so far...
    r/Bitcoin: 20000 posts so far...
    r/Bitcoin: 22000 posts so far...
    r/Bitcoin: 24000 posts so far...
    r/Bitcoin: 26000 posts so far...
    r/Bitcoin: 28000 posts so far...
    r/Bitcoin: 30000 posts so far...
    r/Bitcoin: 32000 posts so far...
    r/Bitcoin: 34000 posts so far...
    r/Bitcoin: 36000 posts so far...
    r/Bitcoin: 38000 posts so far...
    r/Bitcoin: 40000 posts so far...
    r/Bitcoin: 42000 posts so far...
    r/Bitcoin: 44000 posts so far...
    r/Bitcoin: 46000 posts so far...
    r/Bitcoin: 48000 posts so far...
    r/Bitcoin: 50000 posts so fa

---
## 5. ✅ Collection Summary & Data Validation

In [ ]:
print("=" * 60)
print("DATA COLLECTION SUMMARY")
print("=" * 60)

# Check what files were created.
for root, dirs, fls in os.walk("data/raw"):
    for f in sorted(fls):
        fp = os.path.join(root, f)
        size_mb = os.path.getsize(fp) / 1e6
        print(f"  {fp:60s}  {size_mb:8.2f} MB")

print("\n" + "=" * 60)

In [ ]:
# ---- Validate market data ----
print("MARKET DATA VALIDATION")
print("-" * 40)

try:
    mkt = pd.read_csv("data/raw/market/all_assets_daily.csv", parse_dates=["date"])
    for asset in mkt["asset"].unique():
        sub = mkt[mkt["asset"] == asset]
        gaps = sub["date"].diff().dt.days
        max_gap = gaps.max()
        null_prices = sub["price_usd"].isna().sum()
        print(f"  {asset:12s} | {len(sub):5d} days | "
              f"{sub['date'].min().date()} to {sub['date'].max().date()} | "
              f"max gap: {max_gap:.0f}d | null prices: {null_prices}")
    print("\nMarket data looks good!")
except Exception as e:
    print(f"Could not validate: {e}")

In [ ]:
# ---- Validate Reddit data ----
print("REDDIT DATA VALIDATION")
print("-" * 40)

try:
    rdt = pd.read_csv("data/raw/sentiment/reddit/all_reddit_posts.csv")
    print(f"  Total posts: {len(rdt)}")
    print(f"  Date range:  {rdt['date'].min()} to {rdt['date'].max()}")
    print(f"  Subreddits:  {rdt['subreddit'].nunique()}")
    print(f"\n  Per-subreddit:")
    for sub, cnt in rdt["subreddit"].value_counts().items():
        print(f"    r/{sub:25s} {cnt:>8,} posts")
    print("\nReddit data looks good!")
except Exception as e:
    print(f"Could not validate: {e}")

---
## 6. 📥 Download All Collected Data

Run the cell below to zip everything and download it to your computer.

In [ ]:
import shutil

# Zip the entire data directory.
zip_name = "dissertation_collected_data"
shutil.make_archive(zip_name, 'zip', '.', 'data')

zip_path = f"{zip_name}.zip"
size_mb = os.path.getsize(zip_path) / 1e6
print(f"Archive created: {zip_path} ({size_mb:.1f} MB)")
print("\nDownloading...")

files.download(zip_path)
print("Done! Save this zip file and extract it into your dissertation_project/ directory.")

---
## Next Steps

After downloading the collected data:

1. **Data preprocessing** — Clean, align timestamps, handle missing values.
2. **Sentiment analysis** — Run the LLM-based classifier (Ollama) on the text data.
3. **Merge datasets** — Create a unified daily time-series combining market and sentiment data.
4. **Analysis** — VAR, Granger causality, transfer entropy, and predictive modelling.

---
*Notebook created for BASC0024 Final Year Dissertation, UCL Arts and Sciences.*